In [1]:
print("OK")

OK


In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\MediBot-AI\\research'

In [8]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\HP\\Desktop\\MediBot-AI'

In [9]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [10]:
# Extract text from pdf files
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [11]:
extracted_data = load_pdf_files("data")

In [12]:
len(extracted_data)

637

In [18]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append (
            Document(
            page_content=doc.page_content,
            metadata={"source": src}
            )
        )
    return minimal_docs

In [19]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [23]:
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, 
        chunk_overlap=20,
    )
    texts_chunk =  text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [24]:
text_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 5859


In [31]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()

In [33]:
vector = embedding.embed_query("Hello world")


In [34]:
print("Vector length: ", len(vector))

Vector length:  384


In [35]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [37]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [38]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY 
pc = Pinecone(api_key=pinecone_api_key)    

In [39]:
pc

In [41]:
from pinecone import ServerlessSpec

index_name = "medibot-ai"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
index

In [42]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks, 
    embedding=embedding, 
    index_name=index_name
)

In [43]:
# load existing index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embedding, 
    index_name=index_name
)

# Add more data to the existing pinecone index

In [44]:
dswith = Document(
    page_content="This is a sample document.",
    metadata={"source": "sample.pdf"}
)

In [45]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [46]:
retrieved_docs = retriever.invoke("What is acne?")
retrieved_docs

[Document(id='1464410b-6501-4084-b7cf-4d7e0192ed2f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='9465014f-c023-483a-8999-6655a689f397', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='bdac6f16-3441-431b-a6c0-c63464c5ba30', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [47]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [48]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [50]:
system_prompt = (
    "You are an Medical assistent for questions answering tasks."
    "Use the following retrieved documents to answer the question."
    "the question. If you don't know the answer, say you don't know. "
    "Use three sentences at max to answer the question."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt), 
        ("human", "{input}")
    ]
)


In [58]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [59]:
response = rag_chain.invoke({"input" : "What is acne?"})
print(response["answer"])

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}